In [ ]:
import matplotlib
matplotlib.use("Agg")
import torch
from torch import nn
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import gc
import os
import getpass
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
import copy
from torchmetrics.classification import BinaryAccuracy, BinaryPrecision, BinaryRecall
from tqdm.auto import tqdm

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using Device = {str(device).title()}")

script_dir = os.path.dirname(os.path.abspath(os.getcwd()))


df_path = os.path.join(script_dir,"Network Datasets","Financial Risk Dataset.csv")
df = pd.read_csv(df_path)

df = df.dropna()

x = df.drop(columns="is_default").values
encoder = LabelEncoder()
y = encoder.fit_transform(df["is_default"]).reshape(-1,1)

x_scaler = StandardScaler()
x = x_scaler.fit_transform(x)

x_train, x_test, y_train, y_test = train_test_split(x,y,test_size=0.2, random_state=12)
num_features = x_train.shape[1]

x_train = torch.tensor(x_train, dtype=torch.float32).to(device)
y_train = torch.tensor(y_train, dtype=torch.float32).to(device)
x_test = torch.tensor(x_test, dtype=torch.float32).to(device)
y_test = torch.tensor(y_test, dtype=torch.float32).to(device)

user_account = getpass.getuser()

print(f"X = {x.shape}")
print(f"Y = {y.shape}")
print(f"X Train = {x_train.shape}")
print(f"Y Train = {y_train.shape}")

class BinaryClassifier(nn.Module):
    def __init__(self, input_dim=num_features):
        super().__init__()
        self.network = nn.Sequential(
            nn.Linear(in_features=input_dim, out_features=8),
            nn.LeakyReLU(),
            nn.BatchNorm1d(8),

            nn.Linear(in_features=8, out_features=16),
            nn.LeakyReLU(),
            nn.BatchNorm1d(16),
            
            nn.Linear(in_features=16, out_features=28),
            nn.LeakyReLU(),
            nn.BatchNorm1d(28),

            nn.Linear(in_features=28, out_features=47),
            nn.LeakyReLU(),
            nn.BatchNorm1d(47),
            nn.Dropout(p=0.7),   
            
            nn.Linear(in_features=47, out_features=52),
            nn.ELU(),
            nn.BatchNorm1d(52),
            nn.Dropout(p=0.8),   

            nn.Linear(in_features=52, out_features=68),
            nn.ELU(),
            nn.BatchNorm1d(68),
            nn.Dropout(p=0.3),

            nn.Linear(in_features=68, out_features=42),
            nn.LeakyReLU(),
            nn.BatchNorm1d(42),   

            nn.Linear(in_features=42, out_features=21),
            nn.LeakyReLU(),
            nn.BatchNorm1d(21),

            nn.Linear(in_features=21, out_features=1)
        )     

    def forward(self, x:torch.Tensor) -> torch.Tensor:
        return self.network(x)

def run_grid_search_binary(rates_list, epochs_list, seed_val):
    best_loss = float("inf")
    best_state = None
    loss_fn = nn.BCEWithLogitsLoss()
    losses = []
    accuracies = []

    print(f"\n== Search Leaderboard ==")
    print(f"{'Learning Rate':<15} | {'Epochs':<8} | {'Val Loss':<10} | {'Val Acc %':<10}")
    print("=" * 65)

    for lr in rates_list:
        for epoch in epochs_list:
            torch.manual_seed(int(seed_val))
            if torch.cuda.is_available():
                torch.cuda.manual_seed_all(int(seed_val))
            
            model = BinaryClassifier().to(device)   
            optimizer = torch.optim.Adam(model.parameters(), lr=lr)

            for e in tqdm(range(epoch)):
                model.train()
                y_preds = model(x_train)
                loss = loss_fn(y_preds, y_train)     

                optimizer.zero_grad()
                loss.backward()
                optimizer.step()

            model.eval()
            with torch.inference_mode():
                test_logits = model(x_test)
                test_loss = loss_fn(test_logits, y_test).item()
                metric = BinaryAccuracy().to(device)
                prec_metric = BinaryPrecision().to(device)
                rec_metric = BinaryRecall().to(device)
                test_acc = metric(torch.sigmoid(test_logits), y_test).item()
                losses.append(test_loss)
                accuracies.append(test_acc)

            print(f"{lr:<15} | {epoch:<8} | {test_loss:<10.4f} | {test_acc * 100} %")
            if test_loss < best_loss:
                best_loss = test_loss
                best_state = copy.deepcopy(model.state_dict()) 

            del model, optimizer
            gc.collect()       
            if torch.cuda.is_available():
                torch.cuda.empty_cache()

    print(f"\n● Best Loss Achieved = {best_loss:<10.4f}")  
    print(f"● Worst Loss = {max(losses):<10.4f}")     
    print(f"● Mean Accuracy = {sum(accuracies) / len(accuracies):<10.4f}")
    print(f"● Precision Score = {prec_metric(torch.sigmoid(test_logits), y_test).item()}")
    print(f"● Recall Score = {rec_metric(torch.sigmoid(test_logits), y_test).item()}")


    return best_state, best_loss, losses

epochs_list = [380, 900, 710, 460]
rates = [0.01, 0.001, 0.1, 0.2]

for file in [f"C:/Users/{user_account}/OneDrive/Desktop/Financial Risk Status Binary Classifier Network Details.txt"]:
    if os.path.exists(file):
        try:os.remove(file)
        except OSError:pass

seed = float(input("-- Enter a Value for a Seed : ").strip())
best_state, best_loss, losses = run_grid_search_binary(rates, epochs_list, seed)
torch.save(best_state, f"C:/Users/{user_account}/OneDrive/Desktop/Financial Risk Status Binary Classifer Network Parameters")
fresh_model = BinaryClassifier().to(device)
fresh_model.load_state_dict(best_state)
fresh_model.eval()

session_inputs = []
session_predictions = []

statuses = {
    1:"Default",
    0:"Not Default"
}
while True:
    raw_name_input = input("-- Enter Customer Name ( or type 'exit' to stop ): ").strip()
    if raw_name_input.lower() == 'exit' or not raw_name_input:
        print("\nExiting Interactive Loop ...")
        break
        
    user_name = raw_name_input.title()
    age_val = float(input("-- Enter Age : ").strip())
    income_val = float(input("-- Enter Annual Income : ").strip())
    credit_val = float(input("-- Enter Credit Score : ").strip())
    dti_val = float(input("-- Enter Debt - to - Income Ratio ( 0.0 to 1.0 ) : ").strip())
    missed_val = float(input("-- Enter Missed Payments Last Year : ").strip())
    util_val = float(input("-- Enter Card Utilization Ratio ( 0.0 to 1.0 ) : ").strip())

    raw_array = np.array([[age_val, income_val, credit_val, dti_val, missed_val, util_val]])
    scaled_tensor = torch.tensor(x_scaler.transform(raw_array), dtype=torch.float32).to(device)

    with torch.inference_mode():
        logits = fresh_model(scaled_tensor)
        probability = torch.sigmoid(logits).item()
        pred_class_idx = 1 if probability >= 0.5 else 0

    string_prediction = encoder.inverse_transform([pred_class_idx])[0]
    print(f"Predicted Financial Risk Status = {statuses.get(string_prediction)}")   

    session_inputs.append(f"{user_name} ( {credit_val} Credit Score )")
    session_predictions.append(probability * 100)

onedrive = input("-- Do you use One Drive or Not ( Yes / No ) :").strip().title()
if onedrive == "Yes":
    desktop_path = f"C:/Users/{user_account}/OneDrive/Desktop/Financial Risk Status Binary Classifier Network Details.txt"
elif onedrive == "No":
    desktop_path = f"C:/Users/{user_account}/Desktop/Financial Risk Status Binary Classifier Network Details.txt"

try:
    with open(desktop_path, "w", encoding="utf-8") as file:
            file.write(f"== Financial Risk Status Binary Classifier Network Weights & Biases ==\n")   
            file.write(f"\n● Parameter Weights = {fresh_model.network[0].weight[0][0].item():.3f}\n")     
            file.write(f"● Parameter Bias = {fresh_model.network[0].bias[0].item():.3f}\n")     
            file.write(f"● Best Loss = {best_loss:.3f}\n")
            file.write(f"● Worst Loss = {max(losses):.3f}\n")
    print(f"\nSuccessfully Saved the Network Details")
except BaseException as e:
    print(f"Failed to Save Network Details = {e}")    


c:\Users\there\AppData\Local\Programs\Python\Python313\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Using Device = Cuda
X = (15000, 6)
Y = (15000, 1)
X Train = torch.Size([12000, 6])
Y Train = torch.Size([12000, 1])

== Search Leaderboard ==
Learning Rate   | Epochs   | Val Loss   | Val Acc % 


100%|██████████| 380/380 [00:05<00:00, 67.63it/s]


0.01            | 380      | 0.3922     | 81.90000057220459 %


100%|██████████| 900/900 [00:14<00:00, 60.58it/s]


...

0.2             | 900      | 0.4044     | 81.36666417121887 %


100%|██████████| 710/710 [00:09<00:00, 74.01it/s]


0.2             | 710      | 0.3816     | 82.96666741371155 %


100%|██████████| 460/460 [00:06<00:00, 72.55it/s]


0.2             | 460      | 0.3764     | 81.96666836738586 %

● Best Loss Achieved = 0.3764    
● Worst Loss = 0.4214    
● Mean Accuracy = 0.8168    
● Precision Score = 0.7950868606567383
● Recall Score = 0.8695936799049377
Predicted Financial Risk Status = Default
Predicted Financial Risk Status = Default
Predicted Financial Risk Status = Default

Exiting Interactive Loop ...

Successfully Saved the Network Details
